# PCA composicional (versión producción)

Notebook corregido y listo para uso repetible en Colab/Jupyter.
Incluye validaciones, registro de auditoría y exportación de resultados.


In [ ]:
# Dependencias
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import gmean
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
# Configuración de entrada/salida
INPUT_CSV = "colab_uploads/sample.csv"  # Cambiar por tu archivo
OUTPUT_DIR = Path("outputs_pca")
OUTPUT_DIR.mkdir(exist_ok=True)
EPSILON = 1e-6

assert Path(INPUT_CSV).exists(), f"No se encontró el archivo: {INPUT_CSV}"
print(f"Entrada: {INPUT_CSV}")
print(f"Salida: {OUTPUT_DIR.resolve()}")


In [ ]:
# Carga de datos
df_raw = pd.read_csv(INPUT_CSV)
print(f"Filas: {len(df_raw):,} | Columnas: {len(df_raw.columns)}")
display(df_raw.head())


In [ ]:
# Selección robusta de columnas mineralógicas numéricas
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    raise ValueError("No hay columnas numéricas para PCA.")

# Excluir coordenadas típicas si existen
exclude_candidates = {"x", "y", "z", "X", "Y", "Z", "lon", "lat", "depth"}
mineral_cols = [c for c in numeric_cols if c not in exclude_candidates]
if len(mineral_cols) < 2:
    raise ValueError("Se requieren al menos 2 columnas mineralógicas numéricas.")

print("Columnas mineralógicas usadas:")
print(mineral_cols)


In [ ]:
# Limpieza y validación
audit_log = []
def log(msg):
    audit_log.append({"timestamp_utc": datetime.now(timezone.utc).isoformat(), "message": msg})

X = df_raw[mineral_cols].copy()

# Convertir a numérico por seguridad
X = X.apply(pd.to_numeric, errors="coerce")
missing_before = int(X.isna().sum().sum())
if missing_before > 0:
    log(f"Se imputaron {missing_before} valores faltantes con mediana por columna.")
    X = X.fillna(X.median(numeric_only=True))

negatives = int((X < 0).sum().sum())
if negatives > 0:
    raise ValueError(f"Se detectaron {negatives} valores negativos. CLR requiere datos positivos.")

zeros = int((X == 0).sum().sum())
if zeros > 0:
    log(f"Se reemplazaron {zeros} ceros con epsilon={EPSILON}.")
    X = X.replace(0, EPSILON)

display(X.describe().T)


In [ ]:
# Cierre composicional
row_sum = X.sum(axis=1)
if (row_sum <= 0).any():
    raise ValueError("Hay filas con suma <= 0; no es posible el cierre composicional.")

X_closed = X.div(row_sum, axis=0)
max_dev = float((X_closed.sum(axis=1) - 1).abs().max())
print(f"Desviación máxima de cierre (suma fila - 1): {max_dev:.2e}")


In [ ]:
# Transformación CLR
geom = gmean(X_closed, axis=1)
if np.any(geom <= 0):
    raise ValueError("Media geométrica no positiva; revisar datos de entrada.")

X_clr = np.log(X_closed.div(geom, axis=0))
assert np.isfinite(X_clr.to_numpy()).all(), "CLR generó valores no finitos."
display(X_clr.head())


In [ ]:
# Escalado + PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clr)

pca = PCA(n_components=None, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained_ratio = pca.explained_variance_ratio_
eigenvalues = pca.explained_variance_

summary_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_ratio))],
    "Eigenvalue": eigenvalues,
    "Explained_%": explained_ratio * 100,
    "Cumulative_%": np.cumsum(explained_ratio) * 100,
})
display(summary_df)


In [ ]:
# Loadings y scores
loadings = pd.DataFrame(
    pca.components_.T,
    index=mineral_cols,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
)

scores = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    index=df_raw.index,
)

display(loadings.head())
display(scores.head())


In [ ]:
# Visualizaciones
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(range(1, len(eigenvalues) + 1), eigenvalues, marker="o")
ax[0].axhline(1.0, color="red", linestyle="--", label="Kaiser")
ax[0].set_title("Scree plot")
ax[0].set_xlabel("Componente")
ax[0].set_ylabel("Eigenvalue")
ax[0].legend()

sns.scatterplot(data=scores, x="PC1", y="PC2", s=25, alpha=0.75, ax=ax[1])
ax[1].set_title(f"Scores PC1 vs PC2 ({explained_ratio[0]*100:.1f}% / {explained_ratio[1]*100:.1f}%)")
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].axvline(0, color="gray", lw=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# Exportación de artefactos
summary_path = OUTPUT_DIR / "pca_summary.csv"
loadings_path = OUTPUT_DIR / "pca_loadings.csv"
scores_path = OUTPUT_DIR / "pca_scores.csv"
audit_path = OUTPUT_DIR / "audit_log.csv"

summary_df.to_csv(summary_path, index=False)
loadings.to_csv(loadings_path)
scores.to_csv(scores_path, index=True)
pd.DataFrame(audit_log).to_csv(audit_path, index=False)

print("Archivos generados:")
for p in [summary_path, loadings_path, scores_path, audit_path]:
    print(f" - {p}")
